# 📚 SQL Ch.6 — String & Date Functions
> BigQuery SQL Reference Guide, Chapter 6: LIKE · CONCAT · TRIM/UPPER/LOWER · SUBSTR/SPLIT · STRING_AGG · DATE_TRUNC · EXTRACT · DATE_DIFF · FORMAT_DATE  
> BigQuery SQL 완전 참조 가이드 6장: LIKE · CONCAT · TRIM/UPPER/LOWER · SUBSTR/SPLIT · STRING_AGG · DATE_TRUNC · EXTRACT · DATE_DIFF · FORMAT_DATE

📌 **A note on this notebook's SQL engine / 이 노트북의 SQL 엔진에 대한 안내:** This chapter's functions vary more across SQL engines than any other chapter — even the source guide constantly compares BigQuery to PostgreSQL/MySQL/Snowflake here. This notebook runs on **DuckDB**, so a few functions need a DuckDB-specific spelling to actually execute; each one is called out clearly with a 🔧 marker, showing the BigQuery syntax you're learning right next to the DuckDB syntax that runs here.  
이번 챕터의 함수들은 다른 어떤 챕터보다 SQL 엔진마다 차이가 큽니다 — 원본 가이드도 이 챕터에서 BigQuery를 PostgreSQL/MySQL/Snowflake와 끊임없이 비교합니다. 이 노트북은 **DuckDB**로 실행되므로, 일부 함수는 실제로 실행되려면 DuckDB 전용 표기가 필요합니다. 이런 경우는 🔧 표시로 분명히 짚어서, 배우고 있는 BigQuery 문법과 여기서 실행되는 DuckDB 문법을 나란히 보여드립니다.

---
# 🎯 Learning Objective
Today I want to learn: / 오늘 배우고 싶은 것:
- [x] Clean and reshape text with `LIKE`, `TRIM`, `UPPER`/`LOWER`, `SUBSTR`, and `CONCAT`  
`LIKE`, `TRIM`, `UPPER`/`LOWER`, `SUBSTR`, `CONCAT`으로 텍스트를 정제하고 재구성한다
- [x] Group dates into periods with `DATE_TRUNC` and pull out date parts with `EXTRACT`, and explain when to use each  
`DATE_TRUNC`로 날짜를 기간별로 묶고 `EXTRACT`로 날짜 구성요소를 뽑아내며, 각각 언제 써야 하는지 설명한다
- [x] Recognize that string/date function syntax differs meaningfully across SQL engines, and know where to look up the right spelling  
문자열·날짜 함수 문법이 SQL 엔진마다 실질적으로 다르다는 것을 인지하고, 올바른 표기를 어디서 찾아야 하는지 안다

---
# 🧠 Concept

## What is it?
*(Explain it in your own words.)*

**EN:** String functions clean and reshape text (trimming whitespace, fixing case, cutting out substrings, gluing pieces together), while date functions group, extract from, and do arithmetic on dates (rounding a date down to its month, pulling out just the year, computing days between two dates). Both exist because raw data is rarely stored in the exact shape you need it in for analysis.

**KR:** 문자열 함수는 텍스트를 정제하고 재구성하며(공백 제거, 대소문자 정리, 부분 문자열 자르기, 조각들을 붙이기), 날짜 함수는 날짜를 그룹화하고 추출하고 연산합니다(날짜를 월 단위로 내림, 연도만 뽑기, 두 날짜 사이 일수 계산). 원본 데이터는 분석에 필요한 딱 그 모양으로 저장되어 있는 경우가 거의 없기 때문에 둘 다 존재합니다.

## Why do we use it?
*(When is it useful?)*

**EN:** Emails come in mixed case, names carry stray whitespace from copy-paste, and dates need to be grouped by month or quarter for any report that isn't day-by-day. Without these functions, "clean the data" would mean manually editing a spreadsheet row by row — these functions do that cleaning as a repeatable, auditable step inside the query itself.

**KR:** 이메일은 대소문자가 뒤섞여 있고, 이름에는 복사-붙여넣기로 생긴 불필요한 공백이 있으며, 일별이 아닌 리포트는 거의 다 월별이나 분기별로 날짜를 묶어야 합니다. 이 함수들이 없다면 "데이터 정제"는 스프레드시트를 행마다 손으로 편집하는 것을 뜻하게 됩니다 — 이 함수들은 그 정제를 쿼리 안에서 반복 가능하고 검증 가능한 단계로 만들어줍니다.

## When is it used in Business Analytics?
*(Real-world use case)*

**EN:** "Monthly revenue" needs `DATE_TRUNC` + `GROUP BY`. "How long has this customer been with us" needs `DATE_DIFF`. Cleaning a CSV of messy emails/names before a marketing send needs `TRIM`/`LOWER`. This chapter's functions are the unglamorous, constant background work that makes every other chapter's numbers trustworthy.

**KR:** "월별 매출"은 `DATE_TRUNC` + `GROUP BY`가 필요하고, "이 고객이 우리와 함께한 지 얼마나 됐나"는 `DATE_DIFF`가 필요합니다. 마케팅 발송 전 지저분한 이메일/이름이 담긴 CSV를 정제하는 데는 `TRIM`/`LOWER`가 필요합니다. 이번 챕터의 함수들은 화려하진 않지만, 다른 모든 챕터의 숫자를 신뢰할 수 있게 만들어주는 꾸준한 배경 작업입니다.

**Comparison / 비교표:**

| Task / 작업 | SQL (BigQuery) | Pandas |
|---|---|---|
| Pattern match / 패턴 매칭 | `LIKE '%x%'` | `.str.contains("x")` |
| Concatenate / 문자열 연결 | `CONCAT()` / `\|\|` | `+` or f-string |
| Trim whitespace / 공백 제거 | `TRIM()` | `.str.strip()` |
| Change case / 대소문자 변경 | `UPPER()` / `LOWER()` | `.str.upper()` / `.str.lower()` |
| Round down to period / 기간 단위로 내림 | `DATE_TRUNC` | `.dt.to_period()` |
| Extract a component / 구성요소 추출 | `EXTRACT` | `.dt.year` / `.dt.month` |
| Date arithmetic / 날짜 연산 | `DATE_ADD` / `DATE_DIFF` | `+ pd.DateOffset(...)` / subtraction |
| Format as string / 문자열로 포맷 | `FORMAT_DATE` | `.dt.strftime()` |

---
# 📝 Syntax

## Basic Syntax
`LIKE` — pattern matching with two wildcards: `%` (zero or more of any character) and `_` (exactly one of any character).  
`LIKE` — 와일드카드 두 개로 패턴을 매칭: `%`(임의 문자 0개 이상)와 `_`(임의 문자 정확히 1개).

In [1]:
# --- Environment setup / 환경 설정 ---
# We use DuckDB: a free, in-memory SQL engine that understands BigQuery-style syntax
# almost 1:1 (window functions, QUALIFY, ROLLUP, STRING_AGG, etc.), and can query
# pandas DataFrames directly by name -- no separate "load data" step needed.
# DuckDB는 무료 인메모리 SQL 엔진으로, BigQuery 문법(윈도우 함수, QUALIFY, ROLLUP,
# STRING_AGG 등)을 거의 그대로 이해하고, pandas DataFrame을 이름으로 바로 조회할 수
# 있습니다. 별도의 "데이터 로드" 단계가 필요 없습니다.
import duckdb
import pandas as pd
from IPython.display import display

def run(sql: str) -> pd.DataFrame:
    """Execute a SQL string against DuckDB and return the result as a DataFrame.
    SQL 문자열을 DuckDB에서 실행하고 결과를 DataFrame으로 반환합니다."""
    return duckdb.sql(sql).df()

customers = pd.DataFrame({
    "customer_id": ["C01", "C02", "C03", "C04"],
    "name":        ["김민수", "이영희", "박준호", "최서연"],
    "email":       ["kim.minsu@company.com", "lee@gmail.com", "park@company.com", "choi.seo@naver.com"],
})

print("-- % : ends with @company.com --")
display(run("SELECT name, email FROM customers WHERE email LIKE '%@company.com'"))

print("-- % : contains 'gmail' anywhere --")
display(run("SELECT name, email FROM customers WHERE email LIKE '%gmail%'"))
# pandas equivalent: df["email"].str.contains("company"), df["email"].str.endswith("company.com")


-- % : ends with @company.com --


,name,email
0,김민수,kim.minsu@company.com
1,박준호,park@company.com


-- % : contains 'gmail' anywhere --


,name,email
0,이영희,lee@gmail.com


## Common Variations

In [2]:
print("-- _ : exactly one character -- e.g. names that are exactly 3 characters long --")
print("-- _ : 정확히 한 글자 -- 예: 이름이 정확히 3글자인 경우 --")
display(run("SELECT name FROM customers WHERE name LIKE '___'"))

print("-- NOT LIKE: the opposite of a pattern --")
display(run("SELECT name FROM customers WHERE email NOT LIKE '%@company.com'"))

# LIKE is case-sensitive in BigQuery -- and in DuckDB too. '%COMPANY%' will NOT match 'company.com'.
# LIKE는 BigQuery에서 대소문자를 구분함 -- DuckDB도 마찬가지. '%COMPANY%'는 'company.com'과 매칭 안 됨.
r = run("SELECT name FROM customers WHERE email LIKE '%COMPANY%'")
print(f"rows matching uppercase 'COMPANY' pattern: {len(r)}  (0, because LIKE is case-sensitive)")


-- _ : exactly one character -- e.g. names that are exactly 3 characters long --
-- _ : 정확히 한 글자 -- 예: 이름이 정확히 3글자인 경우 --


,name
0,김민수
1,이영희
2,박준호
3,최서연


-- NOT LIKE: the opposite of a pattern --


,name
0,이영희
1,최서연


rows matching uppercase 'COMPANY' pattern: 0  (0, because LIKE is case-sensitive)


---
# 🧪 Small Examples

## Example 1 — CONCAT / \|\|: String Concatenation / 문자열 연결
**EN:** `CONCAT(a, b, c, ...)` glues any number of strings together; the `||` operator does the same thing for exactly two (or chained, more) strings. **In real BigQuery**, if *any* argument to `CONCAT` is `NULL`, the entire result becomes `NULL` — a single missing middle name can wipe out an otherwise-complete full name.  
**KR:** `CONCAT(a, b, c, ...)`는 여러 문자열을 이어 붙이고, `||` 연산자는 두 개(또는 체이닝해서 그 이상)의 문자열에 대해 같은 일을 합니다. **실제 BigQuery에서는** `CONCAT`의 인자 중 *하나라도* `NULL`이면 결과 전체가 `NULL`이 됩니다 — 중간 이름 하나가 없을 뿐인데 멀쩡한 전체 이름이 통째로 사라질 수 있습니다.

🔧 **This notebook's engine (DuckDB):** `CONCAT()` here actually *skips* `NULL` arguments instead of propagating them — a real behavioral difference, not just a syntax one. The `||` operator, however, *does* propagate `NULL` here, matching BigQuery's `CONCAT` behavior exactly. So to see the "one NULL wipes out everything" lesson play out in this notebook, watch the `||` example below, not `CONCAT()`.  
🔧 **이 노트북의 엔진(DuckDB):** 여기서 `CONCAT()`은 `NULL` 인자를 전파하는 대신 실제로 *건너뜁니다* — 단순 문법 차이가 아니라 실제 동작 차이입니다. 반면 `||` 연산자는 여기서 `NULL`을 *그대로 전파*해서 BigQuery의 `CONCAT` 동작과 정확히 일치합니다. 그래서 "NULL 하나가 전체를 지운다"는 교훈을 이 노트북에서 확인하려면 `CONCAT()`이 아니라 아래 `||` 예시를 보세요.

In [3]:
customers1 = pd.DataFrame({
    "first_name": ["민수", "영희", "준호"],
    "last_name":  ["김", "이", "박"],
    "region":     ["서울", "부산", "인천"],
})

sql = """
SELECT
    CONCAT(last_name, first_name) AS full_name,
    CONCAT(region, ' ', last_name, first_name) AS display_name
FROM customers1
"""
display(run(sql))

print("-- || operator: same idea, different syntax --")
display(run("SELECT last_name || first_name AS full_name FROM customers1"))

print("-- NULL behavior comparison, in THIS notebook's engine (DuckDB) --")
print("-- 이 노트북 엔진(DuckDB)에서의 NULL 동작 비교 --")
display(run("SELECT CONCAT('민수', NULL, '김') AS concat_result"))      # DuckDB: skips the NULL / NULL을 건너뜀
display(run("SELECT '민수' || NULL || '김' AS pipe_result"))            # DuckDB: NULL wins, matches BigQuery's CONCAT
# In real BigQuery, CONCAT('민수', NULL, '김') would return NULL, same as the || row above.
# 실제 BigQuery에서는 CONCAT('민수', NULL, '김')도 위 || 행처럼 NULL을 반환함.


,full_name,display_name
0,김민수,서울 김민수
1,이영희,부산 이영희
2,박준호,인천 박준호


-- || operator: same idea, different syntax --


,full_name
0,김민수
1,이영희
2,박준호


-- NULL behavior comparison, in THIS notebook's engine (DuckDB) --
-- 이 노트북 엔진(DuckDB)에서의 NULL 동작 비교 --


,concat_result
0,민수김


,pipe_result
0,<NA>


## Example 2 — TRIM / UPPER / LOWER / LENGTH: String Cleaning / 문자열 정제
**EN:** `TRIM()` strips leading/trailing whitespace (`LTRIM`/`RTRIM` do just one side), `UPPER()`/`LOWER()` normalize case for consistent comparisons, and `LENGTH()` counts characters — often layered together, e.g. `LENGTH(TRIM(name))` to get a *meaningful* length that ignores accidental padding.  
**KR:** `TRIM()`은 앞뒤 공백을 제거하고(`LTRIM`/`RTRIM`은 한쪽만), `UPPER()`/`LOWER()`는 일관된 비교를 위해 대소문자를 통일하며, `LENGTH()`는 글자 수를 셉니다 — 종종 겹쳐서 씁니다, 예: `LENGTH(TRIM(name))`으로 실수로 붙은 공백을 무시한 *의미 있는* 길이를 구합니다.

In [4]:
raw_contacts = pd.DataFrame({
    "customer_id": ["C01", "C02", "C03"],
    "name":        [" 김민수 ", "이영희", " 박준호"],   # note the stray spaces / 불필요한 공백에 주목
    "email":       ["KIM@COMPANY.COM", "lee@Gmail.com", "PARK@company.COM"],
})

sql = """
SELECT
    customer_id,
    TRIM(name)          AS name_clean,
    LOWER(email)         AS email_lower,
    UPPER(email)         AS email_upper,
    LENGTH(TRIM(name))   AS name_length
FROM raw_contacts
"""
display(run(sql))
# pandas equivalent: df["name"].str.strip(), df["email"].str.lower(), df["name"].str.len()


,customer_id,name_clean,email_lower,email_upper,name_length
0,C01,김민수,kim@company.com,KIM@COMPANY.COM,3
1,C02,이영희,lee@gmail.com,LEE@GMAIL.COM,3
2,C03,박준호,park@company.com,PARK@COMPANY.COM,3


## Example 3 — SUBSTR / SPLIT: Position-Based Extraction & Splitting / 부분 추출, 분리
**EN:** `SUBSTR(string, start, length)` extracts a piece of a string by position — `start` is **1-indexed** (the first character is position 1, not 0), and `length` is optional (omit it to go to the end). `SPLIT(string, delimiter)` breaks a string into an array wherever the delimiter appears.  
**KR:** `SUBSTR(문자열, 시작위치, 길이)`는 위치 기준으로 문자열 일부를 추출합니다 — `시작위치`는 **1부터 시작**(첫 글자가 0이 아니라 1)하며, `길이`는 생략 가능(생략하면 끝까지). `SPLIT(문자열, 구분자)`는 구분자가 나올 때마다 문자열을 잘라 배열로 만듭니다.

🔧 **This notebook's engine (DuckDB):** BigQuery accesses a `SPLIT()` array with `[OFFSET(0)]` (0-indexed). DuckDB uses plain brackets with **1-indexed** access instead: `[1]` for the first element, not `[OFFSET(0)]`.  
🔧 **이 노트북의 엔진(DuckDB):** BigQuery는 `SPLIT()` 배열을 `[OFFSET(0)]`(0부터 시작)로 접근합니다. DuckDB는 대신 **1부터 시작**하는 대괄호로 접근합니다: 첫 원소는 `[OFFSET(0)]`이 아니라 `[1]`.

In [5]:
products3 = pd.DataFrame({
    "product_code": ["EL-2024-NB01", "EL-2024-MS02", "CL-2024-SH03"],
    "description":  ["노트북,전자,고급형", "마우스,전자,기본형", "셔츠,의류,일반형"],
})

print("-- SUBSTR: position-based extraction (1-indexed), works exactly like BigQuery --")
sql_substr = """
SELECT
    product_code,
    SUBSTR(product_code, 1, 2) AS dept_code,
    SUBSTR(product_code, 4, 4) AS year_code,
    SUBSTR(product_code, 9)    AS item_code
FROM products3
"""
display(run(sql_substr))

print("-- SPLIT: BigQuery writes arr[OFFSET(0)]; DuckDB (here) writes arr[1] -- both 'first element' --")
print("-- SPLIT: BigQuery는 arr[OFFSET(0)], DuckDB(여기)는 arr[1] -- 둘 다 '첫 번째 원소' --")
sql_split = """
SELECT
    product_code,
    SPLIT(description, ',')[1] AS category,
    SPLIT(description, ',')[2] AS type,
    SPLIT(description, ',')[3] AS grade
FROM products3
"""
display(run(sql_split))
# pandas equivalent: df["description"].str.split(",", expand=True)


-- SUBSTR: position-based extraction (1-indexed), works exactly like BigQuery --


,product_code,dept_code,year_code,item_code
0,EL-2024-NB01,EL,2024,NB01
1,EL-2024-MS02,EL,2024,MS02
2,CL-2024-SH03,CL,2024,SH03


-- SPLIT: BigQuery writes arr[OFFSET(0)]; DuckDB (here) writes arr[1] -- both 'first element' --
-- SPLIT: BigQuery는 arr[OFFSET(0)], DuckDB(여기)는 arr[1] -- 둘 다 '첫 번째 원소' --


,product_code,category,type,grade
0,EL-2024-NB01,노트북,전자,고급형
1,EL-2024-MS02,마우스,전자,기본형
2,CL-2024-SH03,셔츠,의류,일반형


## Example 4 — STRING_AGG: Aggregating Strings Across a Group / 그룹별 문자열 합치기
**EN:** `STRING_AGG(col, separator)` is an aggregate function (used with `GROUP BY`, just like `SUM`/`COUNT`) that combines every value in a group into one delimited string. Add `ORDER BY` inside the function call to control the order the pieces get joined in — without it, the order isn't guaranteed. `NULL` values are skipped automatically.  
**KR:** `STRING_AGG(열, 구분자)`는 (`SUM`/`COUNT`처럼 `GROUP BY`와 함께 쓰는) 집계 함수로, 그룹 안의 모든 값을 하나의 구분자로 이어진 문자열로 합칩니다. 함수 호출 안에 `ORDER BY`를 추가하면 합쳐지는 순서를 통제할 수 있는데, 없으면 순서가 보장되지 않습니다. `NULL` 값은 자동으로 건너뜁니다.

In [6]:
orders_with_products = pd.DataFrame({
    "customer_id":  ["C01", "C01", "C02", "C02", "C02"],
    "product_name": ["노트북", "마우스", "키보드", "마우스", "모니터"],
})

sql = """
SELECT
    customer_id,
    STRING_AGG(product_name, ', ' ORDER BY product_name) AS products_bought,
    COUNT(*) AS purchase_count
FROM orders_with_products
GROUP BY customer_id
"""
display(run(sql))
# pandas equivalent: df.groupby("customer_id")["product_name"].apply(lambda x: ", ".join(sorted(x)))


,customer_id,products_bought,purchase_count
0,C01,"노트북, 마우스",2
1,C02,"마우스, 모니터, 키보드",3


## Example 5 — DATE_TRUNC: Rounding a Date Down to a Period / 기간 단위로 자르기
**EN:** `DATE_TRUNC(date, MONTH)` rounds a date *down* to the start of its containing period — `2024-01-15` and `2024-01-31` both become `2024-01-01`. The result is still a `DATE`, which is exactly what makes it perfect for `GROUP BY`: every date in the same month collapses to the identical value.  
**KR:** `DATE_TRUNC(date, MONTH)`는 날짜를 그 기간의 시작일로 *내림* 처리합니다 — `2024-01-15`와 `2024-01-31` 모두 `2024-01-01`이 됩니다. 결과는 여전히 `DATE` 타입인데, 바로 이 점이 `GROUP BY`에 딱 맞는 이유입니다: 같은 달의 모든 날짜가 동일한 값으로 합쳐집니다.

🔧 **This notebook's engine (DuckDB):** the guide's own callout box already shows BigQuery's `DATE_TRUNC(date, MONTH)` (date first, unit second as a bare keyword) vs. PostgreSQL/Snowflake's `DATE_TRUNC('month', date)` (unit first, as a string). **DuckDB follows the PostgreSQL form** — `date_trunc('month', order_date)` — so that's what runs below.  
🔧 **이 노트북의 엔진(DuckDB):** 원본 가이드의 콜아웃 박스가 이미 BigQuery의 `DATE_TRUNC(date, MONTH)`(날짜 먼저, 단위는 키워드)와 PostgreSQL/Snowflake의 `DATE_TRUNC('month', date)`(단위 먼저, 문자열)를 비교해서 보여줍니다. **DuckDB는 PostgreSQL 방식을 따르므로** — `date_trunc('month', order_date)` — 아래에서 이 형태로 실행됩니다.

In [7]:
orders5 = pd.DataFrame({
    "order_id":   [1001, 1002, 1003, 1004, 1005],
    "order_date": ["2024-01-15", "2024-02-03", "2024-02-21", "2024-03-08", "2024-03-25"],
    "amount":     [45000, 32000, 61000, 28000, 95000],
})
orders5["order_date"] = pd.to_datetime(orders5["order_date"]).dt.date

sql = """
SELECT
    order_date,
    date_trunc('month',   order_date) AS month_start,
    date_trunc('quarter', order_date) AS quarter_start,
    date_trunc('year',    order_date) AS year_start
FROM orders5
"""
display(run(sql))

print("-- the real payoff: GROUP BY month for a monthly total (the most common BA use) --")
print("-- 진짜 활용: GROUP BY month로 월별 합계 (BA가 가장 흔히 쓰는 방식) --")
sql2 = """
SELECT date_trunc('month', order_date) AS month, SUM(amount) AS monthly_total
FROM orders5
GROUP BY month
ORDER BY month
"""
display(run(sql2))


,order_date,month_start,quarter_start,year_start
0,2024-01-15,2024-01-01,2024-01-01,2024-01-01
1,2024-02-03,2024-02-01,2024-01-01,2024-01-01
2,2024-02-21,2024-02-01,2024-01-01,2024-01-01
3,2024-03-08,2024-03-01,2024-01-01,2024-01-01
4,2024-03-25,2024-03-01,2024-01-01,2024-01-01


-- the real payoff: GROUP BY month for a monthly total (the most common BA use) --
-- 진짜 활용: GROUP BY month로 월별 합계 (BA가 가장 흔히 쓰는 방식) --


,month,monthly_total
0,2024-01-01,45000.0
1,2024-02-01,93000.0
2,2024-03-01,123000.0


## Example 6 — EXTRACT: Pulling Out Date Components / 날짜 구성요소 추출
**EN:** `EXTRACT(part FROM date)` pulls a single numeric component out of a date — year, month, day, quarter, day-of-week. Unlike `DATE_TRUNC`'s date-shaped output, `EXTRACT` returns a plain integer, so it's the right tool when you need to *filter or compare* on a date part (`WHERE EXTRACT(MONTH FROM date) = 3`), not group by it.  
**KR:** `EXTRACT(부분 FROM date)`는 날짜에서 숫자 구성요소 하나를 뽑아냅니다 — 연도, 월, 일, 분기, 요일. `DATE_TRUNC`의 날짜 형태 출력과 달리 `EXTRACT`는 순수한 정수를 반환하므로, 날짜 구성요소를 *필터링하거나 비교*할 때(`WHERE EXTRACT(MONTH FROM date) = 3`) 쓰기 알맞은 도구입니다. 그룹화용이 아닙니다.

⚠️ **EN:** BigQuery's `DAYOFWEEK` numbers Sunday=1 through Saturday=7. **In this notebook's engine (DuckDB), `DAYOFWEEK` numbers Sunday=0 through Saturday=6** — one lower across the board. Neither matches the ISO standard (Monday=1). The lesson either way: **always verify day-of-week numbering for your specific engine** before trusting it.  
⚠️ **BigQuery의 `DAYOFWEEK`는 일요일=1부터 토요일=7까지입니다. 이 노트북의 엔진(DuckDB)에서 `DAYOFWEEK`는 일요일=0부터 토요일=6까지**로, 전체적으로 1씩 낮습니다. 둘 다 ISO 표준(월요일=1)과도 다릅니다. 어느 쪽이든 교훈은 같습니다: 신뢰하기 전에 **자신이 쓰는 엔진의 요일 번호 규칙을 항상 확인하세요**.

In [8]:
sql = """
SELECT
    order_date,
    EXTRACT(YEAR    FROM order_date) AS yr,
    EXTRACT(MONTH   FROM order_date) AS mo,
    EXTRACT(DAY     FROM order_date) AS dy,
    EXTRACT(QUARTER FROM order_date) AS qtr,
    EXTRACT(DAYOFWEEK FROM order_date) AS dow
FROM orders5
"""
display(run(sql))
# 2024-01-15 is a Monday: DuckDB's DAYOFWEEK gives 1 here.
# In real BigQuery, the same Monday would show DAYOFWEEK=2 (its scale starts one higher).
# 2024-01-15는 월요일: DuckDB의 DAYOFWEEK는 여기서 1을 줌.
# 실제 BigQuery라면 같은 월요일이 DAYOFWEEK=2로 나옴 (전체적으로 1 높은 척도).

print()
print("-- EXTRACT is a number, so it works directly in WHERE -- March orders only --")
print("-- EXTRACT는 숫자라서 WHERE에서 바로 사용 가능 -- 3월 주문만 --")
display(run("SELECT order_id, order_date FROM orders5 WHERE EXTRACT(MONTH FROM order_date) = 3"))
# pandas equivalent: df["order_date"].dt.year, df["order_date"].dt.month, df["order_date"].dt.dayofweek (0=Monday in pandas!)


,order_date,yr,mo,dy,qtr,dow
0,2024-01-15,2024,1,15,1,1
1,2024-02-03,2024,2,3,1,6
2,2024-02-21,2024,2,21,1,3
3,2024-03-08,2024,3,8,1,5
4,2024-03-25,2024,3,25,1,1



-- EXTRACT is a number, so it works directly in WHERE -- March orders only --
-- EXTRACT는 숫자라서 WHERE에서 바로 사용 가능 -- 3월 주문만 --


,order_id,order_date
0,1004,2024-03-08
1,1005,2024-03-25


## Example 7 — DATE_ADD / DATE_SUB / DATE_DIFF: Date Arithmetic / 날짜 연산
**EN:** `DATE_DIFF(date1, date2, part)` computes `date1 − date2` in the given unit (days, months, years); `DATE_ADD`/`DATE_SUB` shift a date forward or backward by an `INTERVAL`. These turn "how long has this customer been with us" and "what's the date 30 days from now" from manual arithmetic into one function call.  
**KR:** `DATE_DIFF(date1, date2, part)`는 지정한 단위(일, 월, 년)로 `date1 − date2`를 계산하고, `DATE_ADD`/`DATE_SUB`는 날짜를 `INTERVAL`만큼 앞뒤로 옮깁니다. "이 고객이 우리와 함께한 지 얼마나 됐나", "지금부터 30일 뒤 날짜는" 같은 걸 손 계산 대신 함수 호출 하나로 처리하게 해줍니다.

🔧 **This notebook's engine (DuckDB):** BigQuery's `DATE_DIFF(date1, date2, DAY)` and `DATE_ADD(date, INTERVAL 30 DAY)` function forms aren't available as-is. DuckDB uses `date_diff('day', startdate, enddate)` (unit first, as a string) for differences, and plain `+`/`-` with `INTERVAL` for addition/subtraction (no `DATE_ADD(...)` function wrapper needed).  
🔧 **이 노트북의 엔진(DuckDB):** BigQuery의 `DATE_DIFF(date1, date2, DAY)`와 `DATE_ADD(date, INTERVAL 30 DAY)` 함수 형태는 그대로 쓸 수 없습니다. DuckDB는 차이 계산에 `date_diff('day', 시작일, 종료일)`(단위가 먼저, 문자열)을 쓰고, 더하기/빼기는 `DATE_ADD(...)` 함수 없이 그냥 `+`/`-`와 `INTERVAL`을 씁니다.

In [9]:
customers7 = pd.DataFrame({
    "customer_id":  ["C01", "C02", "C03"],
    "name":         ["김민수", "이영희", "박준호"],
    "signup_date":  ["2023-01-10", "2023-06-15", "2024-02-01"],
})
customers7["signup_date"] = pd.to_datetime(customers7["signup_date"]).dt.date

print("-- DATE_DIFF: days/months/years since signup, as of a fixed reference date 2024-06-01 --")
print("-- DATE_DIFF: 기준일 2024-06-01 기준으로 가입 후 경과 일/월/년 --")
sql_diff = """
SELECT
    name,
    signup_date,
    date_diff('day',   signup_date, DATE '2024-06-01') AS days_since_signup,
    date_diff('month', signup_date, DATE '2024-06-01') AS months_since_signup,
    date_diff('year',  signup_date, DATE '2024-06-01') AS years_since_signup
FROM customers7
"""
display(run(sql_diff))
# (In real code you'd likely use CURRENT_DATE() instead of a fixed date -- a literal date
#  is used here so this notebook's output stays the same no matter when you run it.)
# (실제로는 고정 날짜 대신 CURRENT_DATE()를 쓰는 경우가 많음 -- 이 노트북에서는 언제 실행해도
#  결과가 똑같이 나오도록 리터럴 날짜를 사용.)

print()
print("-- DATE_ADD / DATE_SUB equivalent: + / - with INTERVAL --")
sql_add = """
SELECT
    signup_date,
    signup_date + INTERVAL 30 DAY   AS plus_30days,
    signup_date + INTERVAL 3 MONTH  AS plus_3months,
    signup_date - INTERVAL 1 YEAR   AS minus_1year
FROM customers7
WHERE customer_id = 'C01'
"""
display(run(sql_add))
# pandas equivalent: DATE_DIFF = (pd.Timestamp("2024-06-01") - df["signup_date"]).dt.days
#                     DATE_ADD = df["signup_date"] + pd.DateOffset(months=3)


-- DATE_DIFF: days/months/years since signup, as of a fixed reference date 2024-06-01 --
-- DATE_DIFF: 기준일 2024-06-01 기준으로 가입 후 경과 일/월/년 --


,name,signup_date,days_since_signup,months_since_signup,years_since_signup
0,김민수,2023-01-10,508,17,1
1,이영희,2023-06-15,352,12,1
2,박준호,2024-02-01,121,4,0



-- DATE_ADD / DATE_SUB equivalent: + / - with INTERVAL --


,signup_date,plus_30days,plus_3months,minus_1year
0,2023-01-10,2023-02-09,2023-04-10,2022-01-10


## Example 8 — FORMAT_DATE: Formatting a Date as Text / 날짜를 문자열로 포맷
**EN:** `FORMAT_DATE(format_string, date)` turns a `DATE` into a display-ready `STRING` using format codes like `%Y` (4-digit year), `%m` (2-digit month), `%B` (month name), `%A` (weekday name). Because the output is text, not a date, it's for *display* — not for `GROUP BY` (use `DATE_TRUNC` for that) or comparisons (use `EXTRACT`).  
**KR:** `FORMAT_DATE(포맷문자열, date)`는 `%Y`(4자리 연도), `%m`(2자리 월), `%B`(월 영문 이름), `%A`(요일 영문 이름) 같은 포맷 코드를 사용해 `DATE`를 표시용 `STRING`으로 바꿉니다. 출력이 날짜가 아니라 텍스트이므로 *표시용*입니다 — `GROUP BY`(그건 `DATE_TRUNC`)나 비교(그건 `EXTRACT`)에는 쓰지 않습니다.

🔧 **This notebook's engine (DuckDB):** `FORMAT_DATE` doesn't exist here. DuckDB's equivalent is `strftime(date, format_string)` — same format codes (`%Y`, `%m`, `%B`, `%A`, even mixed with literal Korean text), just a different function name **and the two arguments swapped** (date first, format string second).  
🔧 **이 노트북의 엔진(DuckDB):** `FORMAT_DATE`는 여기 없습니다. DuckDB의 대응 함수는 `strftime(date, 포맷문자열)`입니다 — 포맷 코드는 동일(`%Y`, `%m`, `%B`, `%A`, 한글 리터럴과 섞어 써도 동일)하지만, 함수 이름이 다르고 **두 인자의 순서가 뒤바뀝니다**(날짜가 먼저, 포맷 문자열이 나중).

In [10]:
sql = """
SELECT
    order_date,
    strftime(order_date, '%Y-%m')          AS year_month,
    strftime(order_date, '%Y년 %m월 %d일')  AS korean_date,
    strftime(order_date, '%B %d, %Y')       AS english_date,
    strftime(order_date, '%A')              AS weekday_name
FROM orders5
"""
display(run(sql))
# pandas equivalent: df["order_date"].dt.strftime("%Y-%m")  -- pandas uses this same strftime spelling!


,order_date,year_month,korean_date,english_date,weekday_name
0,2024-01-15,2024-01,2024년 01월 15일,"January 15, 2024",Monday
1,2024-02-03,2024-02,2024년 02월 03일,"February 03, 2024",Saturday
2,2024-02-21,2024-02,2024년 02월 21일,"February 21, 2024",Wednesday
3,2024-03-08,2024-03,2024년 03월 08일,"March 08, 2024",Friday
4,2024-03-25,2024-03,2024년 03월 25일,"March 25, 2024",Monday


## Example 9 — Common Combinations / 자주 쓰는 조합
**EN:** **Pattern A** is the BA-essential monthly report shape: `DATE_TRUNC` groups orders by month, then ordinary aggregates (`COUNT`, `COUNTIF`, `SUM` + `CASE WHEN` from Chapter 5) summarize each month. **Pattern B** combines `DATE_DIFF` with `CASE WHEN` to bucket customers into tenure segments — a frequent input to churn or loyalty analysis.  
**KR:** **패턴 A**는 BA에게 필수적인 월별 리포트 형태입니다: `DATE_TRUNC`가 주문을 월별로 묶고, 일반 집계 함수(`COUNT`, `COUNTIF`, 5장의 `SUM` + `CASE WHEN`)가 각 월을 요약합니다. **패턴 B**는 `DATE_DIFF`와 `CASE WHEN`을 조합해 고객을 가입 기간 구간으로 나눕니다 — 이탈·충성도 분석에 자주 쓰이는 입력값입니다.

In [11]:
print("-- Pattern A: DATE_TRUNC + GROUP BY -- monthly revenue report --")
print("-- 패턴 A: DATE_TRUNC + GROUP BY -- 월별 매출 리포트 --")
orders_a = pd.DataFrame({
    "order_id":   [1001, 1002, 1003, 1004, 1005, 1006],
    "order_date": ["2024-01-15", "2024-02-03", "2024-02-21", "2024-03-08", "2024-03-25", "2024-03-30"],
    "status":     ["완료", "완료", "취소", "완료", "완료", "완료"],
    "amount":     [45000, 32000, 61000, 28000, 95000, 42000],
})
orders_a["order_date"] = pd.to_datetime(orders_a["order_date"]).dt.date
sql_a = """
SELECT
    date_trunc('month', order_date) AS month,
    COUNT(*) AS order_count,
    COUNTIF(status = '완료') AS completed_count,
    SUM(CASE WHEN status = '완료' THEN amount ELSE 0 END) AS revenue
FROM orders_a
GROUP BY month
ORDER BY month
"""
display(run(sql_a))

print("-- Pattern B: DATE_DIFF + CASE WHEN -- tenure segment --")
print("-- 패턴 B: DATE_DIFF + CASE WHEN -- 고객 가입 기간별 세그먼트 --")
customers_b = pd.DataFrame({
    "customer_id": ["C01", "C02", "C03"],
    "name":        ["김민수", "이영희", "박준호"],
    "signup_date": ["2022-03-10", "2023-06-15", "2024-05-01"],
})
customers_b["signup_date"] = pd.to_datetime(customers_b["signup_date"]).dt.date
sql_b = """
SELECT
    name,
    signup_date,
    date_diff('day', signup_date, DATE '2024-06-01') AS days_active,
    CASE
        WHEN date_diff('day', signup_date, DATE '2024-06-01') >= 365 THEN '1년 이상'
        WHEN date_diff('day', signup_date, DATE '2024-06-01') >= 90  THEN '90일 이상'
        ELSE '신규(90일 미만)'
    END AS customer_segment
FROM customers_b
"""
display(run(sql_b))


-- Pattern A: DATE_TRUNC + GROUP BY -- monthly revenue report --
-- 패턴 A: DATE_TRUNC + GROUP BY -- 월별 매출 리포트 --


,month,order_count,completed_count,revenue
0,2024-01-01,1,1.0,45000.0
1,2024-02-01,2,1.0,32000.0
2,2024-03-01,3,3.0,165000.0


-- Pattern B: DATE_DIFF + CASE WHEN -- tenure segment --
-- 패턴 B: DATE_DIFF + CASE WHEN -- 고객 가입 기간별 세그먼트 --


,name,signup_date,days_active,customer_segment
0,김민수,2022-03-10,814,1년 이상
1,이영희,2023-06-15,352,90일 이상
2,박준호,2024-05-01,31,신규(90일 미만)


## Example 10 — Practice / 실습 문제
**EN:** Fill in each `________` blank below, then remove the `#` in front of the matching `display(run(...))` line to check your answer. Hints: `TRIM` `LOWER` `date_trunc` `EXTRACT` `'day'`  
**KR:** 아래 `________` 빈칸을 채운 뒤, 해당 `display(run(...))` 줄 앞의 `#`을 지우고 실행해서 답을 확인하세요. 힌트: `TRIM` `LOWER` `date_trunc` `EXTRACT` `'day'`

In [14]:
customers_p = pd.DataFrame({
    "customer_id": ["C01", "C02", "C03"],
    "full_name":   [" 김민수 ", "이영희", "박준호"],
    "email":       ["KIM@COMPANY.COM", "lee@gmail.com", "PARK@NAVER.COM"],
    "signup_date": ["2023-03-15", "2024-01-10", "2023-09-22"],
})
customers_p["signup_date"] = pd.to_datetime(customers_p["signup_date"]).dt.date

# Q1. Lowercase the email, and trim stray whitespace from the name.
# Q1. email을 소문자로 변환하고, 이름의 앞뒤 공백을 제거하라.
q1 = """
SELECT
    TRIM(full_name) AS name_clean,
    LOWER(email) AS email_lower
FROM customers_p
"""
display(run(q1))   # <- uncomment once filled in / 빈칸을 채운 뒤 주석 해제

# Q2. For each customer: month start of signup, signup month number, and days active up to 2025-01-01.
# Q2. 가입일 기준으로: 월 시작일, 가입 월 숫자, 2025-01-01까지의 가입 일수.
# (this notebook's engine: use date_trunc('month', ...) and date_diff('day', start, end))
q2 = """
SELECT
    customer_id,
    DATE_TRUNC('month', signup_date) AS signup_month_start,
    EXTRACT(MONTH FROM signup_date) AS signup_month_num,
    date_diff('day', signup_date, DATE '2025-01-01') AS days_active
FROM customers_p
"""
display(run(q2))   # <- uncomment once filled in / 빈칸을 채운 뒤 주석 해제

# Q3. Count customers and list their (trimmed) names per signup year.
# Q3. 가입 연도별 고객 수와 (공백 제거한) 이름 목록을 조회하라.
q3 = """
SELECT
    EXTRACT(YEAR FROM signup_date) AS signup_year,
    COUNT(*) AS customer_count,
    STRING_AGG(TRIM(full_name), ', ' ORDER BY TRIM(full_name)) AS names
FROM customers_p
GROUP BY signup_year
ORDER BY signup_year
"""
display(run(q3))   # <- uncomment once filled in / 빈칸을 채운 뒤 주석 해제

print("✏️  Fill in the ________ blanks above, uncomment the display() lines, then re-run this cell.")
print("✏️  위 ________ 빈칸을 채우고 display() 줄의 주석을 해제한 뒤 이 셀을 다시 실행하세요.")


,name_clean,email_lower
0,김민수,kim@company.com
1,이영희,lee@gmail.com
2,박준호,park@naver.com


,customer_id,signup_month_start,signup_month_num,days_active
0,C01,2023-03-01,3,658
1,C02,2024-01-01,1,357
2,C03,2023-09-01,9,467


,signup_year,customer_count,names
0,2023,2,"김민수, 박준호"
1,2024,1,이영희


✏️  Fill in the ________ blanks above, uncomment the display() lines, then re-run this cell.
✏️  위 ________ 빈칸을 채우고 display() 줄의 주석을 해제한 뒤 이 셀을 다시 실행하세요.


<details>
<summary>🔑 Answer / 정답 (click to expand / 클릭해서 펼치기)</summary>

```sql
-- Q1
SELECT
    TRIM(full_name) AS name_clean,
    LOWER(email) AS email_lower
FROM customers_p

-- Q2 (this notebook's DuckDB spelling)
SELECT
    customer_id,
    date_trunc('month', signup_date) AS signup_month_start,
    EXTRACT(MONTH FROM signup_date) AS signup_month_num,
    date_diff('day', signup_date, DATE '2025-01-01') AS days_active
FROM customers_p

-- Q3
SELECT
    EXTRACT(YEAR FROM signup_date) AS signup_year,
    COUNT(*) AS customer_count,
    STRING_AGG(TRIM(full_name), ', ' ORDER BY TRIM(full_name)) AS names
FROM customers_p
GROUP BY signup_year
ORDER BY signup_year
```

*(Reference: in real BigQuery, Q2 would be written `DATE_TRUNC(signup_date, MONTH)` and `DATE_DIFF(DATE '2025-01-01', signup_date, DAY)`.)*
*(참고: 실제 BigQuery라면 Q2는 `DATE_TRUNC(signup_date, MONTH)`와 `DATE_DIFF(DATE '2025-01-01', signup_date, DAY)`로 씁니다.)*
</details>

---
# ⚠️ Common Mistakes

**Mistake 1 — Assuming `LIKE` is case-insensitive**
- EN: `WHERE email LIKE '%Company%'` will *not* match `'kim@company.com'` in BigQuery (or in DuckDB) — the comparison is case-sensitive by default.
- KR: `WHERE email LIKE '%Company%'`는 BigQuery(그리고 DuckDB)에서 `'kim@company.com'`과 매칭되지 *않습니다* — 기본적으로 대소문자를 구분합니다.
- ✅ Fix / 해결법: Wrap both sides in `LOWER()` for a case-insensitive match: `WHERE LOWER(email) LIKE '%company%'`.  
대소문자 구분 없는 매칭이 필요하면 양쪽을 `LOWER()`로 감싸세요: `WHERE LOWER(email) LIKE '%company%'`.

**Mistake 2 — Using `DATE_TRUNC` when you actually need `EXTRACT` (or vice versa)**
- EN: `DATE_TRUNC` returns a `DATE` (great for `GROUP BY`, useless for `WHERE month = 3`); `EXTRACT` returns an `INTEGER` (great for filtering/comparing, useless for grouping distinct months across different years — March 2023 and March 2024 both `EXTRACT` to `3`).
- KR: `DATE_TRUNC`는 `DATE`를 반환하고(`GROUP BY`에 좋음, `WHERE month = 3`에는 못 씀), `EXTRACT`는 `INTEGER`를 반환합니다(필터링·비교에 좋음, 연도가 다른 3월들을 구분해서 그룹화하는 데는 못 씀 — 2023년 3월과 2024년 3월 모두 `EXTRACT`하면 `3`).
- ✅ Fix / 해결법: `GROUP BY`/period-bucketing → `DATE_TRUNC`. Filtering/comparing a date part → `EXTRACT`.  
`GROUP BY`·기간 묶기는 `DATE_TRUNC`, 필터링·비교는 `EXTRACT`.

**Mistake 3 — Forgetting that `SUBSTR` positions start at 1, not 0**
- EN: Coming from Python or JavaScript (0-indexed), it's natural to write `SUBSTR(code, 0, 2)` expecting the first two characters — but SQL's `SUBSTR` starts counting at position **1**, so `SUBSTR(code, 0, 2)` actually behaves unexpectedly (often returning one fewer character than intended, depending on the engine).
- KR: Python이나 JavaScript(0부터 시작)에 익숙하면 첫 두 글자를 기대하고 `SUBSTR(code, 0, 2)`라고 쓰기 쉽지만, SQL의 `SUBSTR`은 **1**부터 셉니다. 그래서 `SUBSTR(code, 0, 2)`는 예상과 다르게 동작합니다(엔진에 따라 의도보다 한 글자 적게 나오는 등).
- ✅ Fix / 해결법: Always start position counting at `1` for `SUBSTR`, and remember `SPLIT()` array indexing has its own separate rule per engine (BigQuery: 0-indexed via `OFFSET`; DuckDB: 1-indexed via plain brackets).  
`SUBSTR`은 항상 `1`부터 세고, `SPLIT()` 배열 인덱싱은 엔진마다 별도 규칙이 있다는 것을 기억하세요(BigQuery: `OFFSET`으로 0부터, DuckDB: 대괄호로 1부터).

**Mistake 4 — Assuming string/date function syntax is portable across SQL engines**
- EN: This entire chapter is proof that it isn't — `DATE_TRUNC`'s argument order, `FORMAT_DATE`'s very existence, and day-of-week numbering all vary between BigQuery, PostgreSQL, MySQL, Snowflake, and DuckDB.
- KR: 이번 챕터 전체가 그렇지 않다는 증거입니다 — `DATE_TRUNC`의 인자 순서, `FORMAT_DATE`의 존재 여부, 요일 번호 매기기 모두 BigQuery, PostgreSQL, MySQL, Snowflake, DuckDB 사이에서 다릅니다.
- ✅ Fix / 해결법: When moving a query to a new engine, treat every string/date function as "guilty until proven identical" — check the docs rather than assuming.  
쿼리를 새 엔진으로 옮길 때는 모든 문자열·날짜 함수를 "확인되기 전까지는 다르다"고 취급하세요 — 가정하지 말고 문서를 확인하세요.

---
# 💡 Tips
Useful tips or shortcuts / 유용한 팁과 단축법

- `TRIM` + `LOWER` together is the single most common "make this join actually match" fix — invisible whitespace and case mismatches silently break far more `WHERE`/`JOIN` conditions than people expect.  
 `TRIM` + `LOWER`를 함께 쓰는 것은 "이 조인이 매칭되게 만들기"의 가장 흔한 해결책입니다 — 눈에 안 보이는 공백과 대소문자 불일치는 생각보다 훨씬 자주 `WHERE`/`JOIN` 조건을 조용히 깨뜨립니다.
- Ask yourself "do I need to group by this, or filter/compare on this?" — the answer picks `DATE_TRUNC` or `EXTRACT` for you every time.  
 "이걸로 그룹화할 건가, 아니면 필터링·비교할 건가?"를 자문해 보세요 — 답이 매번 `DATE_TRUNC`와 `EXTRACT` 중 무엇을 쓸지 정해줍니다.
- When you move to a new SQL engine, search "[engine name] date functions cheat sheet" before writing a single date query — five minutes of lookup saves an hour of confused debugging.  
 새 SQL 엔진으로 옮길 때는 날짜 쿼리를 하나라도 쓰기 전에 "[엔진 이름] date functions cheat sheet"를 검색하세요 — 5분의 검색이 1시간의 혼란스러운 디버깅을 아껴줍니다.
- `STRING_AGG` without an inner `ORDER BY` will work, but the piece order isn't guaranteed to be stable — always add `ORDER BY` inside it if the order shown to a stakeholder matters.  
내부 `ORDER BY` 없이 `STRING_AGG`를 써도 동작은 하지만, 조각들의 순서가 안정적으로 보장되지 않습니다 — 이해관계자에게 보여줄 순서가 중요하다면 항상 내부에 `ORDER BY`를 추가하세요.

---
# 🔗 Related Concepts

```
SQL Learning Roadmap (this guide) / SQL 학습 로드맵 (이 가이드)
──────────────────────────────────────────────
 1. SELECT Basics
 2. Aggregation & GROUP BY
 3. JOIN
 4. Subquery & CTE
 5. Conditions & NULL Handling
 6. String & Date Functions      ← ★ YOU ARE HERE / 지금 여기
 7. Window Functions
 8. BA-Specific Patterns
```

```
Match the date function to the job / 작업에 맞는 날짜 함수 고르기
──────────────────────────────────────────────
  "I want to GROUP BY period"        -> DATE_TRUNC   (returns a DATE)
  "기간으로 GROUP BY 하고 싶다"           -> DATE_TRUNC   (DATE 반환)

  "I want to FILTER/COMPARE a part"  -> EXTRACT       (returns an INTEGER)
  "부분을 필터링·비교하고 싶다"            -> EXTRACT       (INTEGER 반환)

  "I want elapsed time between dates"-> DATE_DIFF     (returns an INTEGER)
  "두 날짜 사이 경과 시간이 필요하다"        -> DATE_DIFF     (INTEGER 반환)

  "I want a date shown as text"      -> FORMAT_DATE   (returns a STRING)
  "날짜를 텍스트로 보여주고 싶다"          -> FORMAT_DATE   (STRING 반환)
```

*How is today's topic connected to other concepts?*

**EN:** Pattern A of Example 9 is Chapter 2's `GROUP BY` and Chapter 5's `CASE WHEN`/`COUNTIF`, wearing a `DATE_TRUNC` on top — nothing about grouping or conditional aggregation changed, they just now operate on a *period* instead of a raw value. Looking ahead, Chapter 7 (window functions) will use these same date functions constantly — `DATE_TRUNC` to build a month column, then a window function like `LAG` to compare that month to the previous one, is one of the most common patterns in the entire guide.

**KR:** 예제 9의 패턴 A는 2장의 `GROUP BY`와 5장의 `CASE WHEN`/`COUNTIF` 위에 `DATE_TRUNC`를 얹은 것입니다 — 그룹화나 조건부 집계 자체는 전혀 달라지지 않았고, 그저 원본 값 대신 *기간*을 대상으로 동작할 뿐입니다. 앞으로 배울 7장(윈도우 함수)은 이 날짜 함수들을 끊임없이 사용하는데, `DATE_TRUNC`로 월 열을 만들고 `LAG` 같은 윈도우 함수로 그 달을 전월과 비교하는 것은 이 가이드 전체에서 가장 흔한 패턴 중 하나입니다.

---
# 💼 Business Example
*How would a Business Analyst use this?*

**Scenario / 시나리오:**
**EN:** Marketing hands you a raw customer export before a campaign: names have stray spaces, emails are inconsistently cased, and they want a monthly signup count to plan send volume. *"Can you clean this up and give me signups by month?"*
**KR:** 마케팅팀이 캠페인 전에 지저분한 고객 내보내기 파일을 건넵니다: 이름에는 불필요한 공백이, 이메일은 대소문자가 뒤섞여 있고, 발송량 계획을 위해 월별 가입자 수를 원합니다. *"이거 정리해서 월별 가입자 수 좀 줄 수 있어?"*

**To-do / 할 일:**
- [x] Clean names and emails with `TRIM`/`LOWER`  
`TRIM`/`LOWER`로 이름과 이메일을 정제한다
- [x] Bucket signups into months with `DATE_TRUNC`  
`DATE_TRUNC`로 가입일을 월 단위로 묶는다
- [x] Count signups per month, sorted chronologically  
월별 가입자 수를 세고 시간순으로 정렬한다

In [13]:
raw_export = pd.DataFrame({
    "full_name":   [" 김민수", "이영희 ", "박준호", " 최서연 "],
    "email":       ["KIM@MAIL.COM", "Lee@Mail.com", "park@mail.com", "CHOI@Mail.COM"],
    "signup_date": ["2024-01-08", "2024-01-22", "2024-02-14", "2024-02-27"],
})
raw_export["signup_date"] = pd.to_datetime(raw_export["signup_date"]).dt.date

sql = """
SELECT
    date_trunc('month', signup_date) AS signup_month,
    COUNT(*) AS signup_count
FROM raw_export
GROUP BY signup_month
ORDER BY signup_month
"""
display(run(sql))

print("-- and the cleaned individual rows, for the actual send list --")
print("-- 실제 발송 목록을 위한 개별 정제 행 --")
display(run("SELECT TRIM(full_name) AS name_clean, LOWER(email) AS email_clean, signup_date FROM raw_export"))


,signup_month,signup_count
0,2024-01-01,2
1,2024-02-01,2


-- and the cleaned individual rows, for the actual send list --
-- 실제 발송 목록을 위한 개별 정제 행 --


,name_clean,email_clean,signup_date
0,김민수,kim@mail.com,2024-01-08
1,이영희,lee@mail.com,2024-01-22
2,박준호,park@mail.com,2024-02-14
3,최서연,choi@mail.com,2024-02-27


---
# 📝 Summary
*Write today's concept in 3~5 sentences.*

**EN:** `LIKE` matches text patterns with `%` and `_` wildcards, case-sensitively; `TRIM`/`UPPER`/`LOWER`/`LENGTH` clean up whitespace and casing; `CONCAT`/`||` glue strings together (and in real BigQuery, any `NULL` input poisons a `CONCAT` result entirely); `SUBSTR` extracts by 1-indexed position, and `SPLIT` breaks a string into an array. `STRING_AGG` is the aggregate-function version of concatenation, combining a whole group's values with an optional `ORDER BY` for stable output order. `DATE_TRUNC` rounds a date down to the start of a period (returns a `DATE`, ideal for `GROUP BY`); `EXTRACT` pulls out one numeric component (returns an integer, ideal for filtering); `DATE_DIFF`/`DATE_ADD`/`DATE_SUB` do date arithmetic; `FORMAT_DATE` renders a date as display text. This chapter has more cross-engine syntax variation than any other — BigQuery's exact function names and argument orders don't always transfer directly to other SQL dialects.

**KR:** `LIKE`는 `%`와 `_` 와일드카드로 텍스트 패턴을 대소문자 구분해서 매칭하고, `TRIM`/`UPPER`/`LOWER`/`LENGTH`는 공백과 대소문자를 정리하며, `CONCAT`/`||`는 문자열을 이어붙입니다(실제 BigQuery에서는 `NULL` 입력이 하나라도 있으면 `CONCAT` 결과 전체가 오염됨). `SUBSTR`은 1부터 시작하는 위치로 추출하고, `SPLIT`은 문자열을 배열로 나눕니다. `STRING_AGG`는 연결(concatenation)의 집계 함수 버전으로, 그룹의 모든 값을 합치며 안정적인 출력 순서를 위해 내부에 `ORDER BY`를 추가할 수 있습니다. `DATE_TRUNC`는 날짜를 기간의 시작으로 내림 처리하고(`DATE` 반환, `GROUP BY`에 이상적), `EXTRACT`는 숫자 구성요소 하나를 뽑아내며(정수 반환, 필터링에 이상적), `DATE_DIFF`/`DATE_ADD`/`DATE_SUB`는 날짜 연산을 하고, `FORMAT_DATE`는 날짜를 표시용 텍스트로 만듭니다. 이번 챕터는 다른 어떤 챕터보다 엔진 간 문법 차이가 큽니다 — BigQuery의 정확한 함수 이름과 인자 순서가 다른 SQL dialect로 항상 그대로 옮겨가지는 않습니다.

---
# 📌 One Sentence Summary
Today's topic in ONE sentence. / 오늘 배운 내용을 한 문장으로.

> **EN:** String functions clean text and date functions group/extract/compute on dates — and the biggest lesson of this chapter isn't any single function, it's that their exact spelling and argument order are engine-specific, so always verify rather than assume when you switch SQL dialects.

> **KR:** 문자열 함수는 텍스트를 정제하고 날짜 함수는 날짜를 그룹화·추출·연산하는데, 이번 챕터의 가장 큰 교훈은 어느 특정 함수가 아니라, 정확한 표기와 인자 순서가 엔진마다 다르다는 것입니다 — SQL dialect를 바꿀 때는 항상 가정하지 말고 확인하세요.

---
# ❓ Review Questions

**Q1.** `WHERE email LIKE '%Company%'` finds nothing, even though you know some emails contain "company" in various cases. What's happening, and how do you fix it?  
**Q1.** `WHERE email LIKE '%Company%'`가 아무것도 못 찾았는데, 분명 다양한 대소문자로 "company"가 들어간 이메일이 있다. 무슨 일이 일어난 것이며 어떻게 고치는가?

LIKE is case-sensitive here, so '%Company%' misses 'company'. Fix it with WHERE LOWER(email) LIKE '%company%'.  
대소문자를 통일한 뒤에 LIKE를 걸어야 다양한 표기를 찾을 수 있습니다.

**Q2.** Why does `DATE_TRUNC` return a `DATE` while `EXTRACT` returns an integer — and how does that difference decide which one you use for `GROUP BY` vs. `WHERE`?  
**Q2.** 왜 `DATE_TRUNC`는 `DATE`를 반환하고 `EXTRACT`는 정수를 반환하는가 — 그리고 이 차이가 `GROUP BY`와 `WHERE` 중 무엇을 쓸지 어떻게 결정하는가?

DATE_TRUNC returns a DATE (use it for GROUP BY periods). EXTRACT returns an integer (use it in WHERE to filter a month or year).  
DATE_TRUNC는 DATE를 반환하므로 기간 단위 GROUP BY에 씁니다. EXTRACT는 정수를 반환하므로 WHERE에서 특정 월이나 연도를 걸러낼 때 씁니다.

**Q3.** In real BigQuery, `CONCAT(first_name, middle_name, last_name)` returns `NULL` for a customer with no middle name. Name two different ways to prevent that.  
**Q3.** 실제 BigQuery에서 `CONCAT(first_name, middle_name, last_name)`은 중간 이름이 없는 고객에게 `NULL`을 반환한다. 이를 방지하는 서로 다른 두 가지 방법을 말해보라.

COALESCE(middle_name, '') inside CONCAT, or CONCAT_WS which skips NULLs. Either keeps the rest of the name.  
빈칸으로 채우거나, NULL을 건너뛰는 연결 함수를 쓰면 전체 이름이 사라지지 않습니다.

**Q4.** `SUBSTR(product_code, 1, 2)` and `SUBSTR(product_code, 0, 2)` are not the same thing in SQL. Why not, and which one is correct for "the first two characters"?  
**Q4.** SQL에서 `SUBSTR(product_code, 1, 2)`와 `SUBSTR(product_code, 0, 2)`는 같지 않다. 왜 그런가, 그리고 "첫 두 글자"를 위해서는 어느 쪽이 맞는가?

SQL SUBSTR is 1-indexed, so SUBSTR(col, 1, 2) is the first two characters. Position 0 is not the start.  
SQL의 SUBSTR은 1부터 세므로, 첫 두 글자는 SUBSTR(col, 1, 2)입니다. 위치 0은 시작이 아닙니다.

**Q5.** You're moving a query from BigQuery to a different SQL engine, and it uses `DATE_TRUNC`, `DATE_DIFF`, and `FORMAT_DATE`. What should you check before assuming the query will run unchanged?  
**Q5.** BigQuery 쿼리를 다른 SQL 엔진으로 옮기는 중인데, `DATE_TRUNC`, `DATE_DIFF`, `FORMAT_DATE`를 사용한다. 쿼리가 그대로 동작할 거라고 가정하기 전에 무엇을 확인해야 하는가?

Before assuming the query ports, check each date/string function’s name and argument order in the new engine. DATE_TRUNC, DATE_DIFF, and FORMAT_DATE are especially not portable.  
쿼리가 그대로 옮겨질 거라고 가정하기 전에, 새 엔진에서 각 문자열·날짜 함수의 이름과 인자 순서를 확인해야 합니다. 특히 DATE_TRUNC, DATE_DIFF, FORMAT_DATE는 엔진 간에 이식되지 않는 경우가 많습니다.

---
*📅 Try answering these again in a few days. / 며칠 후 다시 답해보세요.*